In [2]:
import os
os.chdir('..')

In [3]:
import torch
import numpy as np
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.models import CTCModel

/home/dimalkevich/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Обучение c нуля

In [61]:
!python train_model.py --config-name ebranchformer_ctc

/home/dimalkevich/science/asr/speech_rec_course/asr/train_model.py:24: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="conf", config_name="conformer_ctc")
/home/dimalkevich/miniconda3/envs/asr/lib/python3.10/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/next/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read h

# Oбучение с предобученного чекпоинта

In [63]:
!python train_model.py --config-name ebranchformer_ctc init_weights=/home/dimalkevich/science/asr/speech_rec_course/asr/data/checkpoints_and_tokenizer/ebranchformer_ckpt.ckpt

/home/dimalkevich/science/asr/speech_rec_course/asr/train_model.py:24: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="conf", config_name="conformer_ctc")
/home/dimalkevich/miniconda3/envs/asr/lib/python3.10/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/next/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
init
[2024-10-24 18:22:09,331][lightning][INFO] - successful load initial weights
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' |

In [16]:
cfg = OmegaConf.load("./conf/ebranchformer_ctc.yaml")

In [72]:
model = CTCModel(cfg)
model.eval()
model.freeze()

ckpt = torch.load('./data/checkpoints_and_tokenizer/ebranchformer_ckpt.ckpt', map_location='cpu')
model.load_state_dict(ckpt)

<All keys matched successfully>

In [73]:
device = torch.device('cuda')

model.to(device)
model.device

device(type='cuda', index=0)

# Проверить, что у реализованного энкодера число параметров совпадает со значением в статье

In [68]:
print(sum(p.numel() for p in model.parameters()))

11266850


# Посчитать Word Error Rate на датасетах 

In [25]:
from src.metrics import WER

def calculate_greedy_wer(preds_list, encoded_len_list, targets_list):
    metric = WER()
    refs = []
    hyps = []
    for pred_batch, encoded_len_batch, targets_batch in tqdm(zip(preds_list, encoded_len_list, targets_list), total=len(preds_list)):
        for pred, pred_len, target in zip(pred_batch, encoded_len_batch, targets_batch):
            hyps.append(model.decoder.decode_hypothesis(pred[:pred_len], unique_consecutive=True))
            refs.append(model.decoder.decode_hypothesis(target, unique_consecutive=False))
    
    metric.update(refs, hyps)
    wer = metric.compute()
    return wer[0]

## CROWD

In [75]:
features_list, features_len_list = [], []
targets_list, target_len_list = [], []
logprobs_list, encoded_len_list, preds_list = [], [], []

# Отрежем себе кусок валидации для экспериментов
for i, batch in tqdm(enumerate(model.val_dataloader())):
    features, features_len, targets, target_len = batch
    with torch.inference_mode():
        logprobs, encoded_len, preds = model.forward(features.to(device), features_len.to(device))
    features_list.append(features)
    features_len_list.append(features_len)
    targets_list.append(targets)
    target_len_list.append(target_len)
    logprobs_list.append(logprobs)
    encoded_len_list.append(encoded_len)
    preds_list.append(preds)
    if i == 5:
        break

5it [00:10,  2.08s/it]


In [76]:
wer = calculate_greedy_wer(preds_list, encoded_len_list, targets_list)
print('WER: ', wer)

100%|██████████| 6/6 [00:11<00:00,  1.97s/it]

WER:  tensor(1.0094)


## Farfield

In [88]:
cfg.val_dataloader.dataset.manifest_name = "test_opus/farfield/manifest.jsonl"

In [17]:
model = CTCModel(cfg)
model.eval()
model.freeze()

ckpt = torch.load('./data/checkpoints_and_tokenizer/ebranchformer_ckpt.ckpt', map_location='cpu')
model.load_state_dict(ckpt)

<All keys matched successfully>

In [18]:
device = torch.device('cuda')

model.to(device)
model.device

device(type='cuda', index=0)

In [91]:
features_list, features_len_list = [], []
targets_list, target_len_list = [], []
logprobs_list, encoded_len_list, preds_list = [], [], []

# Отрежем себе кусок валидации для экспериментов
for i, batch in tqdm(enumerate(model.val_dataloader())):
    features, features_len, targets, target_len = batch
    with torch.inference_mode():
        logprobs, encoded_len, preds = model.forward(features.to(device), features_len.to(device))
    features_list.append(features)
    features_len_list.append(features_len)
    targets_list.append(targets)
    target_len_list.append(target_len)
    logprobs_list.append(logprobs)
    encoded_len_list.append(encoded_len)
    preds_list.append(preds)
    if i == 5:
        break

5it [00:06,  1.28s/it]


In [92]:
wer = calculate_greedy_wer(preds_list, encoded_len_list, targets_list)
print('WER: ', wer)

100%|██████████| 6/6 [00:07<00:00,  1.19s/it]

WER:  tensor(1.)


# Дообучение

In [45]:
!python train_model.py --config-name ebranchformer_ctc init_weights=/home/dimalkevich/science/asr/speech_rec_course/asr/data/checkpoints_and_tokenizer/ebranchformer_ckpt.ckpt

/home/dimalkevich/science/asr/speech_rec_course/asr/train_model.py:24: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="conf", config_name="conformer_ctc")
/home/dimalkevich/miniconda3/envs/asr/lib/python3.10/site-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/next/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
init
[2024-10-25 17:27:00,335][lightning][INFO] - successful load initial weights
wandb: Currently logged in as: malkevich-dim. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.18.3
wandb: Run data is saved locally in ./wandb/run-20241025_172700-y1sazv1d
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run jumping-cosmos-2
wandb: ⭐️ View project

In [23]:
features_list, features_len_list = [], []
targets_list, target_len_list = [], []
logprobs_list, encoded_len_list, preds_list = [], [], []

# Отрежем себе кусок валидации для экспериментов
for i, batch in tqdm(enumerate(model.val_dataloader())):
    features, features_len, targets, target_len = batch
    with torch.inference_mode():
        logprobs, encoded_len, preds = model.forward(features.to(device), features_len.to(device))
    features_list.append(features)
    features_len_list.append(features_len)
    targets_list.append(targets)
    target_len_list.append(target_len)
    logprobs_list.append(logprobs)
    encoded_len_list.append(encoded_len)
    preds_list.append(preds)
    if i == 5:
        break

5it [00:10,  2.11s/it]


In [26]:
wer = calculate_greedy_wer(preds_list, encoded_len_list, targets_list)
print('WER: ', wer)

100%|██████████| 6/6 [00:10<00:00,  1.83s/it]


WER:  tensor(1.0010)


## НЕ ВЫШЛО(((